# Generative Backend — Unified Action Pipeline

Demonstrates the universal `ActionPipeline` handling **both PickUp and Place**
from a single entry point without knowing the action type in advance.

**Key difference from `01_pick_up_pipeline.ipynb`:**
- One pipeline class, any action type
- One slot-filling LLM call that *classifies* the action **and** extracts parameters
- An `ActionDispatcher` registry routes to the correct handler underneath

**Stages shown:**
1. **Unified slot filling** — one LLM call classifies the instruction and fills a `UnifiedSlotSchema`
2. **PickUp full run** — end-to-end `PickUpAction` via `pipeline.run()`
3. **Place full run** — end-to-end `PlaceAction` via the same `pipeline.run()`
4. **Step-by-step breakdown** — `classify_and_extract()` → `dispatch()` for both actions

**Prerequisites:** `cd cognitive_robot_abstract_machine && uv sync --active`  
**API key:** set `OPENAI_API_KEY` in `generative_backend/.env`

## 1 · Environment & Imports

In [1]:
import logging
import pathlib
from dotenv import load_dotenv

# Load API keys before any LLM import
_env = pathlib.Path().resolve().parent / ".env"
load_dotenv(_env, override=True)

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")

# ── SDT / pycram stack ────────────────────────────────────────────────────────
from semantic_digital_twin.robots.pr2 import PR2
from pycram.testing import setup_world

# ── Generative backend ────────────────────────────────────────────────────────
from generative_backend.action_pipeline import ActionPipeline
from generative_backend.workflows.schemas.pick_up import PickUpSlotSchema
from generative_backend.workflows.schemas.place import PlaceSlotSchema

print("All imports OK")

/home/malineni/envs/cram-env/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Found these qp solvers: ['qpSWIFT', 'qpalm']


WARNING polytope.solvers: `polytope` failed to import `cvxopt.glpk`.
WARNING polytope.solvers: will use `scipy.optimize.linprog`


All imports OK


## 2 · Load World

`setup_world()` loads the apartment URDF, a PR2 robot, a milk carton (`milk.stl`),
and a breakfast cereal box. The milk sits on the kitchen counter.

We will use:
- **Pick-up instruction** — grab the milk from the counter
- **Place instruction** — put the milk on the table

In [2]:
world = setup_world()

try:
    robot = PR2.from_world(world)
except Exception as exc:
    print(f"Note: PR2 full setup skipped ({type(exc).__name__}) — recovering annotation")
    robot = next((a for a in world.semantic_annotations if isinstance(a, PR2)), None)
    if robot is None:
        raise RuntimeError("Could not obtain PR2 annotation") from exc
    with world.modify_world():
        robot._setup_velocity_limits()
        robot._setup_hardware_interfaces()
        robot._setup_joint_states()

# Confirm key bodies are present
milk_body = world.get_body_by_name("milk.stl")
milk_pos  = milk_body.global_pose.to_position()
print(f"World loaded")
print(f"  milk  → x={float(milk_pos.x):.2f}, y={float(milk_pos.y):.2f}, z={float(milk_pos.z):.2f}")

# Show all bodies so we can pick an appropriate place target
all_body_names = [
    str(getattr(getattr(b, "name", None), "name", b))
    for b in world.bodies
]
print(f"  bodies present ({len(all_body_names)}): {', '.join(all_body_names[:20])}")

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
INFO semantic_digital_twin.world: Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO semantic_digital_twin.world: Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO semantic_digital_twin.world: Trying to add kinematic_structure_entity with name pr2/base_link
INFO semantic_digital_twin.world: Trying to add kinematic_structure_entity with name pr2/base_link
INFO semantic_digital_twin.world: Trying to add kinematic_structure_entity with name pr2/base_bellow_link
INFO semantic_digital_twin.world: Trying to add kinematic_structure_entity with name pr2/base_link
INFO semantic_digital_twin.world: Trying to add kinemat

Note: PR2 full setup skipped (WorldEntityNotFoundError) — recovering annotation
World loaded
  milk  → x=2.37, y=2.00, z=1.05
  bodies present (212): apartment_root, furnitures_root, walls, windows, wall_coloksu_wall1, wall_coloksu_wall2, wall_coloksu_wall3, wall_coloksu_wall4, wardrobe, wardrobe_door_left, wardrobe_door_left_handle, wardrobe_door_right, wardrobe_door_right_handle, armchair, sofa, coffee_table, coffee_table_drawer, bedside_table, kitchen_root, side_A


## 3 · Initialise Unified Pipeline

`ActionPipeline.from_world()` auto-detects the robot manipulator and builds
one pipeline object that handles **any** registered action type.

In [3]:
pipeline = ActionPipeline.from_world(world)

print("ActionPipeline ready")
print(f"  manipulator : {type(pipeline.world_context.manipulator).__name__}")

# Show which action types are registered in the dispatcher
from generative_backend.action_dispatcher import ActionDispatcher
print(f"  registered action types: {list(ActionDispatcher._registry.keys())}")

ActionPipeline ready
  manipulator : ParallelGripper
  registered action types: ['PickUpAction', 'PlaceAction']


## 4 · Unified Slot Filling — Classification in Action

A **single LLM call** classifies the instruction into an action type *and* extracts
all relevant parameters at once.  The `action_type` field is the discriminator that
tells the `ActionDispatcher` which handler to invoke.

Run the same call on a pick-up and a place instruction to see the difference.

In [4]:
PICKUP_INSTRUCTION = "Pick up the milk from the counter"

pickup_schema: PickUpSlotSchema = pipeline.classify_and_extract(PICKUP_INSTRUCTION)

print(f"Instruction : {PICKUP_INSTRUCTION!r}")
print(f"Schema type : {type(pickup_schema).__name__}  ← typed by LLM classification")
print()
print("PickUpSlotSchema:")
print(f"  action_type          : {pickup_schema.action_type}")
print(f"  object.name          : {pickup_schema.object_description.name}")
print(f"  object.semantic_type : {pickup_schema.object_description.semantic_type}")
print(f"  object.spatial       : {pickup_schema.object_description.spatial_context}")
print(f"  arm                  : {pickup_schema.arm}   ← null = unspecified")
print(f"  grasp_params         : {pickup_schema.grasp_params}   ← null = unspecified")

Instruction : 'Pick up the milk from the counter'
Schema type : PickUpSlotSchema  ← typed by LLM classification

PickUpSlotSchema:
  action_type          : PickUpAction
  object.name          : milk
  object.semantic_type : FoodItem
  object.spatial       : from the counter
  arm                  : None   ← null = unspecified
  grasp_params         : None   ← null = unspecified


In [5]:
pickup_schema.object_description

EntityDescriptionSchema(name='milk', semantic_type='FoodItem', spatial_context='from the counter', attributes=None)

In [6]:
PLACE_INSTRUCTION = "Place the milk on the table"

place_schema: PlaceSlotSchema = pipeline.classify_and_extract(PLACE_INSTRUCTION)

print(f"Instruction : {PLACE_INSTRUCTION!r}")
print(f"Schema type : {type(place_schema).__name__}  ← typed by LLM classification")
print()
print("PlaceSlotSchema:")
print(f"  action_type               : {place_schema.action_type}")
print(f"  object.name               : {place_schema.object_description.name}")
print(f"  object.semantic_type      : {place_schema.object_description.semantic_type}")
print(f"  arm                       : {place_schema.arm}   ← null = unspecified")
print(f"  target.name               : {place_schema.target_description.name}")
print(f"  target.semantic_type      : {place_schema.target_description.semantic_type}")
print(f"  target.spatial_context    : {place_schema.target_description.spatial_context}")

Instruction : 'Place the milk on the table'
Schema type : PlaceSlotSchema  ← typed by LLM classification

PlaceSlotSchema:
  action_type               : PlaceAction
  object.name               : milk
  object.semantic_type      : FoodItem
  arm                       : None   ← null = unspecified
  target.name               : table
  target.semantic_type      : Table
  target.spatial_context    : None


## 5 · PickUp — Full Pipeline Run

`pipeline.run()` handles the entire chain:
1. Unified slot filling (classifies as `PickUpAction`)
2. Entity grounding → resolves `"milk"` to the `milk.stl` world body
3. `PickUpActionHandler` builds a `PartialDesignator[PickUpAction]`
4. `HybridPickUpResolver` fills arm, approach_direction, vertical_alignment, rotate_gripper via LLM

In [7]:
pickup_action = pipeline.run(PICKUP_INSTRUCTION)

gd = pickup_action.grasp_description
obj_name = str(getattr(getattr(pickup_action.object_designator, "name", None), "name",
                        pickup_action.object_designator))

print(f"PickUpAction resolved from: {PICKUP_INSTRUCTION!r}")
print()
print(f"  object_designator  : {obj_name}")
print(f"  arm                : {pickup_action.arm}")
print(f"  approach_direction : {gd.approach_direction}")
print(f"  vertical_alignment : {gd.vertical_alignment}")
print(f"  rotate_gripper     : {gd.rotate_gripper}")
print(f"  manipulator        : {type(gd.manipulator).__name__}")

Partial Desginator : {'type': 'PickUpAction', 'object_designator': Body(name=PrefixedName('None/milk.stl'), id=UUID('d8dc7df7-c48d-4d4c-a37d-2851f8a47342'), index=113), 'arm': None, 'grasp_description': None}
World Context for pickup action : 
 Robot position: x=1.500, y=2.500, z=0.051
Object 'milk.stl': x=2.370, y=2.000, z=1.050
  → 0.87m in front of and 0.50m to the right of the robot, 1.00m above robot origin.
  → Bounding box (w×d×h): 0.064 × 0.065 × 0.194 m
LLM reasoning for missing params : The object is located 0.87m in front and 0.50m to the right of the robot, making the RIGHT arm the most suitable for reach. The FRONT approach direction provides the best clearance, and since the object is flat and resting on a surface, TOP alignment is preferred with no need to rotate the gripper.
PickUpAction resolved from: 'Pick up the milk from the counter'

  object_designator  : milk.stl
  arm                : RIGHT
  approach_direction : ApproachDirection.FRONT
  vertical_alignment : Ve

In [8]:
type(pickup_action)

pycram.robot_plans.actions.core.pick_up.PickUpAction

## 6 · Place — Full Pipeline Run

The **same** `pipeline.run()` call now handles a place instruction:
1. Unified slot filling classifies as `PlaceAction`
2. `PlaceActionHandler` grounds **two** entities: the object and the target surface
3. LLM resolves which arm to use based on target surface position relative to robot
4. Returns a fully specified `PlaceAction`

In [9]:
place_action = pipeline.run(PLACE_INSTRUCTION)

obj_name = str(getattr(getattr(place_action.object_designator, "name", None), "name",
                        place_action.object_designator))
tgt_name = str(getattr(getattr(place_action.target_location, "name", None), "name",
                        place_action.target_location))

print(f"PlaceAction resolved from: {PLACE_INSTRUCTION!r}")
print()
print(f"  object_designator  : {obj_name}")
print(f"  arm                : {place_action.arm}")
print(f"  target_location    : {tgt_name}")

WARNING generative_backend.action_dispatcher: Grounding warning: Grounding for 'table' returned 4 candidates: ['coffee_table', 'coffee_table_drawer', 'bedside_table', 'table_area_main']. All passed to PartialDesignator.


Partial Desginator : {'type': 'PlaceAction', 'object_designator': Body(name=PrefixedName('None/milk.stl'), id=UUID('d8dc7df7-c48d-4d4c-a37d-2851f8a47342'), index=113), 'target_location': Body(name=PrefixedName('apartment/coffee_table'), id=UUID('6cc21ee5-d791-4292-8688-502d34fea422'), index=15), 'arm': None}
world context for place :  Robot position: x=1.500, y=2.500, z=0.051
Object to place 'milk.stl': x=2.370, y=2.000, z=1.050
Target surface 'apartment/coffee_table': x=16.653, y=2.780, z=0.000
  → 15.15m in front of and 0.28m to the left of the robot.
Target surface 'apartment/coffee_table_drawer': x=16.653, y=2.780, z=0.000
  → 15.15m in front of and 0.28m to the left of the robot.
Target surface 'apartment/bedside_table': x=16.061, y=1.880, z=0.000
  → 14.56m in front of and 0.62m to the right of the robot.
Target surface 'apartment/table_area_main': x=5.000, y=4.000, z=0.000
  → 3.50m in front of and 1.50m to the left of the robot.
LLM reasoning for place :  The target surface is 

## 7 · Step-by-Step: Classify → Dispatch

`pipeline.run()` is a convenience wrapper.  The two underlying steps can be
called separately for debugging or interactive inspection:

```
schema = pipeline.classify_and_extract(instruction)   # Phase 1
action = pipeline.dispatch(schema)                    # Phase 2 → N
```

This also makes it easy to inspect the schema before committing to execution,
or to override fields before dispatching.

In [ ]:
# ── Step 1: classify + extract ─────────────────────────────────────────────────
schema = pipeline.classify_and_extract(PICKUP_INSTRUCTION)
print(f"classify_and_extract({PICKUP_INSTRUCTION!r})")
print(f"  → action_type = {schema.action_type}")
print(f"  → object      = {schema.object_description.name!r}")
print()

# (optional) inspect / override before dispatching
# e.g. schema.arm = "LEFT"  # force left arm

# ── Step 2: dispatch to handler ───────────────────────────────────────────────
action = pipeline.dispatch(schema)
gd = action.grasp_description
print(f"dispatch(schema) → {type(action).__name__}")
print(f"  arm                : {action.arm}")
print(f"  approach_direction : {gd.approach_direction}")
print(f"  vertical_alignment : {gd.vertical_alignment}")
print(f"  rotate_gripper     : {gd.rotate_gripper}")

In [ ]:
# ── Step 1: classify + extract ─────────────────────────────────────────────────
schema = pipeline.classify_and_extract(PLACE_INSTRUCTION)
print(f"classify_and_extract({PLACE_INSTRUCTION!r})")
print(f"  → action_type  = {schema.action_type}")
print(f"  → object       = {schema.object_description.name!r}")
print(f"  → target       = {schema.target_description.name!r}")
print()

# ── Step 2: dispatch to handler ───────────────────────────────────────────────
action = pipeline.dispatch(schema)
obj_name = str(getattr(getattr(action.object_designator, "name", None), "name",
                        action.object_designator))
tgt_name = str(getattr(getattr(action.target_location, "name", None), "name",
                        action.target_location))
print(f"dispatch(schema) → {type(action).__name__}")
print(f"  object_designator  : {obj_name}")
print(f"  arm                : {action.arm}")
print(f"  target_location    : {tgt_name}")

In [ ]:
action

## 8 · Both Actions Back-to-Back

To make the contrast clear: the same three lines of code work for any instruction,
regardless of whether it's a pick-up, a place, or any other registered action type.

In [ ]:
instructions = [
    "Pick up the milk from the counter",
    "Place the milk on the table",
]

for instr in instructions:
    action = pipeline.run(instr)
    print(f"Instruction : {instr!r}")
    print(f"  → {type(action).__name__}")
    print(f"     arm = {action.arm}")
    if hasattr(action, "grasp_description") and action.grasp_description:
        gd = action.grasp_description
        print(f"     approach_direction = {gd.approach_direction}")
        print(f"     vertical_alignment = {gd.vertical_alignment}")
        print(f"     rotate_gripper     = {gd.rotate_gripper}")
    if hasattr(action, "target_location") and action.target_location is not None:
        tgt = str(getattr(getattr(action.target_location, "name", None), "name",
                          action.target_location))
        print(f"     target_location    = {tgt}")
    print()